# TEP — De la predicción a la recomendación de ajustes de línea

Objetivo de este notebook: no solo predecir cómo se comporta el proceso (lo que ya sabes hacer con XGBoost), sino **recomendar qué ajuste de variables manipuladas (XMV) lleva a una mejor variable de calidad (XMEAS)**.

Esto es deliberadamente el mismo tipo de problema que vas a tener con la almazara: variables manipulables de línea (setpoints) → variables medidas de proceso → una variable de calidad final. El dataset y los nombres de columna van a cambiar, pero la arquitectura del pipeline no.

**Para usar el TEP real:**
1. Descarga `TEP_FaultFree_Training.csv` de [kaggle.com/datasets/afrniomelo/tep-csv](https://www.kaggle.com/datasets/afrniomelo/tep-csv)
2. Filtra una sola `simulationRun` (p. ej. `df[df['simulationRun'] == 1]`)
3. Cambia `RUTA_CSV`, `VARIABLES_XMV` y `VARIABLES_XMEAS` en la celda de configuración de más abajo por los nombres reales (`xmv_1`...`xmv_11`, `xmeas_1`...`xmeas_41`)

El resto del notebook funciona igual sin tocar nada más — esa es la idea: la arquitectura no depende del dataset concreto.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import itertools

plt.style.use('ggplot')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

## 1. Configuración — cambia esto para usar tus propios datos

Esta celda es la única que deberías necesitar tocar al cambiar de dataset (simulado → TEP real → datos de la almazara). Todo lo demás está escrito en términos de estas variables.

In [ ]:
RUTA_CSV = '../Datasets/TEP_FaultFree_Training.csv'

# Columna de orden temporal (equivalente a 'sample' en TEP)
COLUMNA_TIEMPO = 'sample'

# La diferencia entre dos samples en tiempo es de 3 minutos

df_raw = pd.read_csv(RUTA_CSV)
df_raw = df_raw.sort_values(COLUMNA_TIEMPO).reset_index(drop=True)

X_dict = {
'XMEAS_1':'A_feed_stream',
'XMEAS_2':'D_feed_stream',
'XMEAS_3':'E_feed_stream',
'XMEAS_4':'Total_fresh_feed_stripper',
'XMEAS_5':'Recycle_flow_into_rxtr',
'XMEAS_6':'Reactor_feed_rate',
'XMEAS_7':'Reactor_pressure',
'XMEAS_8':'Reactor_level',
'XMEAS_9':'Reactor_temp',
'XMEAS_10':'Purge_rate',
'XMEAS_11':'Separator_temp',
'XMEAS_12':'Separator_level',
'XMEAS_13':'Separator_pressure',
'XMEAS_14':'Separator_underflow',
'XMEAS_15':'Stripper_level',
'XMEAS_16':'Stripper_pressure',
'XMEAS_17':'Stripper_underflow',
'XMEAS_18':'Stripper_temperature',
'XMEAS_19':'Stripper_steam_flow',
'XMEAS_20':'Compressor_work',
'XMEAS_21':'Reactor_cooling_water_outlet_temp',
'XMEAS_22':'Condenser_cooling_water_outlet_temp',
'XMEAS_23':'Composition_of_A_rxtr_feed',
'XMEAS_24':'Composition_of_B_rxtr_feed',
'XMEAS_25':'Composition_of_C_rxtr_feed',
'XMEAS_26':'Composition_of_D_rxtr_feed',
'XMEAS_27':'Composition_of_E_rxtr_feed',
'XMEAS_28':'Composition_of_F_rxtr_feed',
'XMEAS_29':'Composition_of_A_purge',
'XMEAS_30':'Composition_of_B_purge',
'XMEAS_31':'Composition_of_C_purge',
'XMEAS_32':'Composition_of_D_purge',
'XMEAS_33':'Composition_of_E_purge',
'XMEAS_34':'Composition_of_F_purge',
'XMEAS_35':'Composition_of_G_purge',
'XMEAS_36':'Composition_of_H_purge',
'XMEAS_37':'Composition_of_D_product',
'XMEAS_38':'Composition_of_E_product',
'XMEAS_39':'Composition_of_F_product',
'XMEAS_40':'Composition_of_G_product',
'XMEAS_41':'Composition_of_H_product',
'XMV_1':'D_feed_flow_valve',
'XMV_2':'E_feed_flow_valve',
'XMV_3':'A_feed_flow_valve',
'XMV_4':'Total_feed_flow_stripper_valve',
'XMV_5':'Compressor_recycle_valve',
'XMV_6':'Purge_valve',
'XMV_7':'Separator_pot_liquid_flow_valve',
'XMV_8':'Stripper_liquid_product_flow_valve',
'XMV_9':'Stripper_steam_valve',
'XMV_10':'Reactor_cooling_water_flow_valve',
'XMV_11':'Condenser_cooling_water_flow_valve',
'XMV_12':'Agitator_speed'
   }

# df_raw = df_raw.rename(columns = lambda x:X_dict[x.upper()] if x.upper() in X_dict.keys()  else x)

print("Dimensiones:", df_raw.shape)
df_raw.head()

# Obtener las filas que pertenecen a la simulación 1
df_sim = df_raw[df_raw['simulationRun'] == 1]

## 2. Cargar datos y vistazo rápido

In [ ]:
# Usa X_dict para renombrar las columnas de df_raw a nombres más descriptivos. Esto facilita la interpretación de los datos y la identificación de las variables relevantes para el análisis.

# Variable de calidad objetivo — lo que quieres optimizar (análogo a acidez/fenoles)
VARIABLE_CALIDAD = 'xmeas_7'

# Variables manipulables (lo que un operador podría ajustar) — tus "XMV"
VARIABLES_XMV = [meas for meas in df_sim.columns if meas.upper() in X_dict.keys() and meas.startswith('xmv_') and meas != VARIABLE_CALIDAD]

# Variables de proceso medidas (sensores) — tus "XMEAS" intermedias
VARIABLES_XMEAS = [meas for meas in df_sim.columns if meas.upper() in X_dict.keys() and meas.startswith('xmeas_') and meas != VARIABLE_CALIDAD]

print("Variables de calidad objetivo:", VARIABLE_CALIDAD)
print("Variables manipulables:", VARIABLES_XMV)
print("Variables de proceso medidas:", VARIABLES_XMEAS)


In [ ]:
fig, axes = plt.subplots(len(VARIABLES_XMV) + len(VARIABLES_XMEAS) + 1, 1,
                          figsize=(14, 2.2 * (len(VARIABLES_XMV) + len(VARIABLES_XMEAS) + 1)),
                          sharex=True)

todas_vars = VARIABLES_XMV + VARIABLES_XMEAS + [VARIABLE_CALIDAD]
colores = plt.cm.tab10(np.linspace(0, 1, len(todas_vars)))

for ax, var, color in zip(axes, todas_vars, colores):
    ax.plot(df_sim[COLUMNA_TIEMPO], df_sim[var], color=color)
    ax.set_ylabel(var, fontsize=8)

axes[-1].set_xlabel(COLUMNA_TIEMPO)
plt.suptitle('Variables manipuladas (XMV), medidas (XMEAS) y calidad')
plt.tight_layout()
plt.show()

## 3. Diagnóstico de señales — antes de meter nada al modelo

Esta sección es puro diagnóstico: no descarta ninguna señal por su cuenta. Te da tres vistas distintas para que **tú** decidas qué quitar, porque el criterio de qué sensor es prescindible depende de conocimiento del proceso que el código no tiene (en la almazara, un operario puede saber que una señal aparentemente redundante es la fiable y la otra la que se ensucia).

Tres preguntas distintas, tres herramientas distintas — no las confundas:

1. **¿Hay señales gemelas (redundantes)?** → matriz de correlación.
2. **¿Qué forma tiene cada señal?** → estadísticos descriptivos.
3. **¿Alguna señal deriva con el tiempo (tendencia)?** → pendiente + r² sobre el índice temporal.

### Nota importante sobre XGBoost y señales correlacionadas

Al contrario que en una regresión lineal, a XGBoost la multicolinealidad **no le rompe las predicciones**: si dos sensores son casi idénticos, el árbol elige uno para el split e ignora el otro, y el RMSE apenas cambia. Entonces, ¿por qué mirar esto? Por dos razones que NO son la precisión:

- **Interpretabilidad**: con dos señales gemelas, XGBoost reparte la importancia entre ambas de forma casi aleatoria — así que tu gráfico de importancia de variables te miente sobre cuál sensor importa de verdad. Quitar la redundante limpia esa lectura.
- **Coste y robustez**: menos columnas = modelo más rápido y menos sensible a que un sensor falle en producción.

Traducción: **no esperes que descartar señales gemelas baje tu RMSE.** Probablemente se quede igual. Lo que mejora es tu capacidad de *entender* el modelo, no su precisión.

In [ ]:
# --- 1. Correlación: detectar señales gemelas ---
señales = VARIABLES_XMEAS + VARIABLES_XMV
corr = df_sim[señales].corr()

# Pares con correlación muy alta (candidatos a redundancia). Umbral orientativo: |r| > 0.95
umbral_corr = 0.95
pares_altos = []
for i in range(len(señales)):
    for j in range(i + 1, len(señales)):
        r = corr.iloc[i, j]
        if abs(r) > umbral_corr:
            pares_altos.append({'señal_A': señales[i], 'señal_B': señales[j], 'correlación': r})

df_pares = pd.DataFrame(pares_altos).sort_values('correlación', key=abs, ascending=False) if pares_altos else pd.DataFrame(columns=['señal_A','señal_B','correlación'])
print(f"Pares de señales con |correlación| > {umbral_corr}: {len(df_pares)}")
print("(candidatos a redundancia -- decide TÚ cuál de cada par conservar, según cuál sea más fiable físicamente)")
df_pares.head(20)

### ¿Colapsar un par correlacionado, o no? Depende de si la relación es instantánea o tiene retardo

Correlación alta entre una XMV y una XMEAS no significa que sean intercambiables como fuente de información — significa que una es probablemente la causa de la otra (el operador mueve la válvula, el sensor lo refleja). Descartar la XMEAS "porque la XMV ya lo dice" solo tiene sentido si esa relación es **instantánea**. Si hay retardo o inercia entre ambas (algo típico en procesos físicos reales), la XMEAS lleva información que la XMV sola no tiene: cuánto ha avanzado ya el efecto, no solo qué se ordenó.

La forma de comprobarlo, no de suponerlo: **correlación cruzada con desfase (lagged cross-correlation)**. Desplazamos una señal respecto a la otra varios pasos en ambas direcciones y vemos en qué desfase la correlación es máxima.

- Si el mejor desfase es 0 (o ±1): la relación es prácticamente instantánea → el par es un buen candidato a colapsar, quedándote con la XMV (permite además hacer prescripción sobre ella).
- Si el mejor desfase es > 1: hay retardo/inercia real → **no** descartes la XMEAS solo por la correlación agregada; aporta información temporal que la XMV no tiene.

In [ ]:
def correlacion_cruzada_maxima(serie_a, serie_b, max_lag=10):
    """
    Compara serie_a con serie_b desplazada entre -max_lag y +max_lag pasos.
    Devuelve el lag donde la correlación es máxima en valor absoluto, y su valor.
    lag > 0 significa que serie_b va POR DETRÁS de serie_a (reacciona con retardo).
    """
    a = serie_a.values if hasattr(serie_a, 'values') else np.asarray(serie_a)
    b = serie_b.values if hasattr(serie_b, 'values') else np.asarray(serie_b)
    resultados = []
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            xa, xb = a[-lag:], b[:lag]
        elif lag > 0:
            xa, xb = a[:-lag], b[lag:]
        else:
            xa, xb = a, b
        if len(xa) > 1 and np.std(xa) > 0 and np.std(xb) > 0:
            r = np.corrcoef(xa, xb)[0, 1]
            resultados.append((lag, r))
    mejor_lag, mejor_r = max(resultados, key=lambda x: abs(x[1]))
    return mejor_lag, mejor_r


# Aplicar solo a los pares XMV-XMEAS de alta correlación ya detectados arriba
# (nos interesan específicamente los pares control-monitorización, no XMEAS-XMEAS)
pares_control_monitorizacion = df_pares[
    df_pares.apply(lambda r: (r['señal_A'] in VARIABLES_XMV) != (r['señal_B'] in VARIABLES_XMV), axis=1)
].copy() if len(df_pares) > 0 else df_pares.copy()

resultados_desfase = []
for _, row in pares_control_monitorizacion.iterrows():
    lag, r_max = correlacion_cruzada_maxima(df_sim[row['señal_A']], df_sim[row['señal_B']], max_lag=10)
    resultados_desfase.append({
        'señal_A': row['señal_A'], 'señal_B': row['señal_B'],
        'correlación_instantánea': row['correlación'],
        'mejor_lag': lag, 'correlación_máxima': r_max,
        'recomendación': 'colapsable (relación ~instantánea)' if abs(lag) <= 1 else f'NO colapsar -- retardo de {abs(lag)} pasos'
    })

df_desfase = pd.DataFrame(resultados_desfase)
print(f"Pares control-monitorización analizados: {len(df_desfase)}")
df_desfase

### Perfil completo de correlación cruzada — no solo el mejor lag, todos

Acotado a los pares que ya salieron como candidatos a alta correlación en la sección anterior (no el cubo 52×52×lags completo — no lo necesitas, ya sabes por sentido físico qué pares importan). Rango de lags 0-5, coherente con `HORIZONTE_FORECAST` (5 pasos = 15 minutos en el TEP real, sample cada 3 min) — si cambias ese horizonte, cambia también `MAX_LAG_CUBO` aquí para que sigan siendo comparables.

Esto es estrictamente una tabla 2D (pares × lags), no un cubo — evitamos deliberadamente calcular relaciones para las que ya sabes, por conocimiento del proceso, que no tienen sentido (variables de unidades distintas del TEP, o desfases más allá de lo que tu horizonte de interés justifica).

In [ ]:
MAX_LAG_CUBO = 5  # mantener coherente con HORIZONTE_FORECAST

def perfil_correlacion_cruzada(serie_a, serie_b, max_lag=MAX_LAG_CUBO):
    """
    Igual que correlacion_cruzada_maxima, pero devuelve la correlación en CADA lag
    del rango, no solo la máxima -- para poder inspeccionar la forma completa del
    perfil (¿sube gradualmente hasta un pico, o es un salto brusco en un único lag?).
    """
    a = serie_a.values if hasattr(serie_a, 'values') else np.asarray(serie_a)
    b = serie_b.values if hasattr(serie_b, 'values') else np.asarray(serie_b)
    perfil = {}
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            xa, xb = a[-lag:], b[:lag]
        elif lag > 0:
            xa, xb = a[:-lag], b[lag:]
        else:
            xa, xb = a, b
        if len(xa) > 1 and np.std(xa) > 0 and np.std(xb) > 0:
            perfil[lag] = np.corrcoef(xa, xb)[0, 1]
        else:
            perfil[lag] = np.nan
    return perfil


# Tabla: filas = pares control-monitorización ya detectados, columnas = cada lag
filas_cubo = []
for _, row in pares_control_monitorizacion.iterrows():
    perfil = perfil_correlacion_cruzada(df_sim[row['señal_A']], df_sim[row['señal_B']])
    fila = {'par': f"{row['señal_A']} - {row['señal_B']}"}
    fila.update({f'lag_{lag:+d}': r for lag, r in perfil.items()})
    filas_cubo.append(fila)

df_cubo = pd.DataFrame(filas_cubo).set_index('par')
print(f"Tabla de correlación cruzada: {df_cubo.shape[0]} pares × {df_cubo.shape[1]} lags")
df_cubo

### ¿El lag 0 es física real del proceso, o un artefacto de esta simulación concreta?

Todo lo anterior se calculó sobre `df_sim`, que es **una sola `simulationRun` (la nº 1)**. Si muchos pares dan pico en lag 0, hay dos explicaciones que solo se distinguen repitiendo el análisis en más experimentos:

- **Es física real**: a la resolución de muestreo del TEP (3 min), las dinámicas del proceso son rápidas comparadas con ese intervalo, así que casi todo parece instantáneo. Si esto se repite de forma consistente en varios experimentos, es una conclusión robusta sobre el proceso.
- **Es un artefacto de esta simulación**: quizá `simulationRun=1` es un tramo especialmente estable, con poca variación — y con series poco dinámicas, cualquier par de señales puede dar correlación alta en cualquier lag por pura coincidencia estadística (correlación espuria). Si el mejor lag cambia mucho de un experimento a otro, tu conclusión de "todo es lag 0" no era sobre el proceso, era sobre esa muestra concreta.

Repetimos el cálculo del mejor lag para los mismos pares, mirando ahora **varias `simulationRun` distintas** (no todas las 500 -- no lo necesitas para responder esta pregunta, un puñado ya te dice si el patrón es estable o no).

In [ ]:
N_SIMULACIONES_A_COMPARAR = 8  # un puñado basta para ver si el patrón es estable; no hace falta las 500

simulaciones_disponibles = sorted(df_raw['simulationRun'].unique())
simulaciones_muestra = simulaciones_disponibles[:N_SIMULACIONES_A_COMPARAR]

resultados_multi_run = []
for run in simulaciones_muestra:
    df_run = df_raw[df_raw['simulationRun'] == run]
    for _, row in pares_control_monitorizacion.iterrows():
        lag, r_max = correlacion_cruzada_maxima(df_run[row['señal_A']], df_run[row['señal_B']], max_lag=MAX_LAG_CUBO)
        resultados_multi_run.append({
            'par': f"{row['señal_A']} - {row['señal_B']}",
            'simulationRun': run,
            'mejor_lag': lag,
            'r_max': r_max
        })

df_multi_run = pd.DataFrame(resultados_multi_run)

# Resumen: para cada par, ¿cuánto varía el mejor lag entre simulaciones?
resumen_estabilidad = df_multi_run.groupby('par').agg(
    lag_medio=('mejor_lag', 'mean'),
    lag_std=('mejor_lag', 'std'),
    lag_moda=('mejor_lag', lambda x: x.mode().iloc[0]),
    r_medio=('r_max', 'mean')
).reset_index()

resumen_estabilidad['estable'] = resumen_estabilidad['lag_std'] < 1.0  # umbral orientativo: menos de 1 lag de dispersión
resumen_estabilidad = resumen_estabilidad.sort_values('lag_std', ascending=False)

print(f"Comparación sobre {len(simulaciones_muestra)} simulaciones ({simulaciones_muestra})")
print("lag_std alto (dispersión grande entre simulaciones) = sospecha de artefacto, no física real")
print("lag_std ~0 (o consistentemente bajo) = el mejor lag se repite -> conclusión robusta")
resumen_estabilidad

In [ ]:
# Visualizar cómo varía el mejor lag de cada par entre simulaciones -- un vistazo rápido
# a qué pares son consistentes (física real) y cuáles saltan de una simulación a otra (ruido)
if len(resumen_estabilidad) > 0:
    fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * df_multi_run['par'].nunique())))
    pares_ordenados = resumen_estabilidad.sort_values('lag_std')['par'].tolist()

    for i, par in enumerate(pares_ordenados):
        lags_par = df_multi_run[df_multi_run['par'] == par]['mejor_lag'].values
        ax.scatter(lags_par, [i] * len(lags_par), alpha=0.6, color='steelblue')

    ax.set_yticks(range(len(pares_ordenados)))
    ax.set_yticklabels(pares_ordenados, fontsize=8)
    ax.set_xlabel('mejor lag encontrado en cada simulación')
    ax.axvline(0, color='gray', linestyle=':', alpha=0.5)
    ax.set_title('Estabilidad del mejor lag entre simulaciones\n(puntos agrupados = consistente; puntos dispersos = posible artefacto)')
    plt.tight_layout()
    plt.show()
else:
    print("No hay pares que comparar.")

In [ ]:
if len(df_cubo) > 0:
    fig, ax = plt.subplots(figsize=(10, max(3, 0.4 * len(df_cubo))))
    im = ax.imshow(df_cubo.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(df_cubo.columns)))
    ax.set_xticklabels(df_cubo.columns, rotation=45, fontsize=8)
    ax.set_yticks(range(len(df_cubo)))
    ax.set_yticklabels(df_cubo.index, fontsize=8)
    plt.colorbar(im, ax=ax, label='Correlación')
    plt.title('Perfil de correlación cruzada por par y lag\n(pico desplazado del centro = relación con retardo/inercia real)')
    plt.tight_layout()
    plt.show()
else:
    print("No hay pares control-monitorización con correlación instantánea alta que perfilar.")

In [ ]:
# Heatmap de correlación para ver la estructura global de un vistazo
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(señales)))
ax.set_yticks(range(len(señales)))
ax.set_xticklabels(señales, rotation=90, fontsize=6)
ax.set_yticklabels(señales, fontsize=6)
plt.colorbar(im, ax=ax, label='Correlación de Pearson')
plt.title('Matriz de correlación entre señales\n(bloques rojos intensos fuera de la diagonal = grupos de señales que se mueven juntas)')
plt.tight_layout()
plt.show()

### 3bis. Correlación de cada señal contra la variable objetivo (con desfase)

Esto responde a una pregunta distinta de la sección 4bis (que mira la estructura temporal de cada señal *en sí misma*): aquí miramos qué tan relacionada está cada señal con `VARIABLE_CALIDAD`, para tener una primera idea de qué candidatas pesan más antes de entrenar nada.

Reutilizamos `perfil_correlacion_cruzada` (ya definida arriba) en vez de solo correlación instantánea -- si la calidad reacciona con retardo a alguna XMEAS, la correlación en el mismo instante puede salir baja aunque la relación real sea fuerte, exactamente el mismo problema que ya vimos con los pares XMV-XMEAS.

**Dos avisos importantes para no sobre-interpretar esta tabla:**
- Es correlación **lineal** (Pearson). XGBoost puede aprovechar relaciones no lineales que esta tabla no va a detectar -- una señal con correlación baja aquí puede seguir siendo importante para el modelo. Esta tabla es una criba rápida previa, no sustituye mirar la importancia de variables del modelo ya entrenado.
- Correlación alta con la calidad no distingue causa de consecuencia -- una XMEAS puede estar correlacionada con la calidad simplemente porque ambas responden a la misma XMV, no porque una cause la otra.

In [ ]:
MAX_LAG_TARGET = 10  # rango de desfase a explorar contra la variable de calidad

resultados_corr_target = []
for var in VARIABLES_XMEAS + VARIABLES_XMV:
    if var == VARIABLE_CALIDAD:
        continue
    perfil = perfil_correlacion_cruzada(df_sim[var], df_sim[VARIABLE_CALIDAD], max_lag=MAX_LAG_TARGET)
    mejor_lag = max(perfil, key=lambda k: abs(perfil[k]) if not np.isnan(perfil[k]) else 0)
    resultados_corr_target.append({
        'señal': var,
        'mejor_lag': mejor_lag,
        'correlación_máxima': perfil[mejor_lag],
        'correlación_instantánea': perfil[0]
    })

df_corr_target = pd.DataFrame(resultados_corr_target)
df_corr_target = df_corr_target.sort_values('correlación_máxima', key=abs, ascending=False).reset_index(drop=True)

print(f"Top 15 señales más correlacionadas (linealmente) con {VARIABLE_CALIDAD}:")
df_corr_target.head(15)

In [ ]:
top_n = 15
df_top = df_corr_target.head(top_n)

fig, ax = plt.subplots(figsize=(9, max(3, 0.35*top_n)))
colores_barra = ['crimson' if v < 0 else 'steelblue' for v in df_top['correlación_máxima']]
ax.barh(df_top['señal'][::-1], df_top['correlación_máxima'][::-1], color=colores_barra[::-1])
ax.set_xlabel('Correlación máxima (con su mejor lag)')
ax.set_title(f'Señales más correlacionadas con {VARIABLE_CALIDAD}\n(azul = positiva, rojo = negativa; recuerda: correlación lineal, no toda la historia)')
ax.axvline(0, color='gray', linestyle=':')
plt.tight_layout()
plt.show()

In [ ]:
# --- 2. Estadísticos descriptivos: la FORMA de cada señal ---
# Ojo: describe() NO detecta tendencia. Una señal que sube y baja puede tener la misma
# media que una plana. Esto te dice el rango y la dispersión, no la dinámica temporal.
estadisticos = df_sim[señales].describe().T
estadisticos['rango'] = estadisticos['max'] - estadisticos['min']
estadisticos['cv'] = estadisticos['std'] / estadisticos['mean'].abs()  # coef. de variación: dispersión relativa
estadisticos.sort_values('cv', ascending=False).head(15)

In [ ]:
# --- 3. Detección de tendencia: ¿alguna señal deriva con el tiempo? ---
# Esto SÍ es una propiedad temporal (a diferencia de describe()). Una señal con tendencia
# fuerte es no-estacionaria, y eso importa para tu split cronológico: si un sensor deriva
# sin parar, el rango de valores del test puede quedar fuera de lo que vio el train, y el
# árbol (que no extrapola) predecirá mal ahí. Ese problema es MÁS grave que la redundancia.
from scipy import stats

def analizar_tendencia(serie):
    x = np.arange(len(serie))
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, serie)
    rango = serie.max() - serie.min()
    # cambio_total_vs_rango ~1 => la tendencia recorre casi todo el rango de la señal (deriva fuerte)
    # ~0 => la señal es plana en promedio (aunque oscile)
    pendiente_norm = abs(slope) * len(serie) / rango if rango > 0 else 0
    return pd.Series({'pendiente_por_paso': slope, 'cambio_total_vs_rango': pendiente_norm, 'r2': r_value**2})

tendencias = df_sim[señales].apply(analizar_tendencia).T
tendencias = tendencias.sort_values('cambio_total_vs_rango', ascending=False)

print("Señales ordenadas por fuerza de tendencia (arriba = más deriva temporal):")
print("  cambio_total_vs_rango cercano a 1 y r2 alto => tendencia clara, revisa si es un problema para tu split")
print("  valores bajos => señal sin deriva sistemática")
tendencias.head(15)

## 4. Autocorrelación — cuántos lags tienen sentido aquí

Igual que hicimos con el dataset de la bomba, pero esta vez con un proceso de dinámica más lenta (más inercia). Espera ver el PACF de la variable de calidad cayendo dentro de la banda mucho más tarde que en el caso de la bomba — eso confirma que aquí sí puede tener sentido un rango de lags más largo, o incluso el espaciado por potencias que descartamos antes.

In [ ]:
from statsmodels.tsa.stattools import acf, pacf
import time

n_lags_analisis = 5
variables_pacf = VARIABLES_XMEAS + [VARIABLE_CALIDAD]

# --- Fase 1: solo cálculo numérico (rápido -- ~0.02s para 40+ variables) ---
# plot_acf/plot_pacf calculan Y dibujan a la vez; con muchas variables el dibujo de una
# figura gigante es lo que realmente tarda (varios segundos por tight_layout + renderizado),
# no el cálculo. Aquí separamos: primero los números, rápido y con feedback inmediato
# variable a variable; el dibujo (Fase 2, más abajo) es opcional y solo para las que
# decidas mirar de cerca.
resultados_pacf = {}
t0_total = time.time()

for var in variables_pacf:
    t0 = time.time()
    valores_acf = acf(df_sim[var], nlags=n_lags_analisis)
    valores_pacf = pacf(df_sim[var], nlags=n_lags_analisis, method='ywm')
    t1 = time.time()

    resultados_pacf[var] = {'acf': valores_acf, 'pacf': valores_pacf}

    # feedback inmediato por variable, sin esperar a que terminen todas
    print(f"{var:30s}  ({t1-t0:.4f}s)")
    print(f"  ACF  (lags 1-{n_lags_analisis}):  " + "  ".join(f"{v:+.3f}" for v in valores_acf[1:]))
    print(f"  PACF (lags 1-{n_lags_analisis}):  " + "  ".join(f"{v:+.3f}" for v in valores_pacf[1:]))

print(f"\nTiempo total ({len(variables_pacf)} variables): {time.time()-t0_total:.3f}s")

### Fase 2 (opcional) — dibujar solo las variables que te interesan

Con los números de arriba ya tienes lo esencial. Dibujar las 40+ gráficas de golpe es lo que causaba la lentitud original (una figura de más de 1 metro de alto, con `tight_layout()` y el renderizado tardando varios segundos). Aquí eliges cuántas y cuáles mirar de cerca — por defecto, las que tienen el PACF más alto en el lag 1 (más autocorrelación = más candidatas a necesitar varios lags como feature).

In [ ]:
# Ranking de variables por fuerza de autocorrelación (PACF en lag 1), para elegir cuáles mirar
ranking = pd.DataFrame([
    {'variable': var, 'pacf_lag1': abs(res['pacf'][1])}
    for var, res in resultados_pacf.items()
]).sort_values('pacf_lag1', ascending=False).reset_index(drop=True)

print("Top 10 variables con mayor autocorrelación parcial en lag 1:")
ranking.head(10)

In [ ]:
N_VARIABLES_A_DIBUJAR = 5  # cambia esto según cuántas quieras inspeccionar visualmente

variables_a_dibujar = ranking.head(N_VARIABLES_A_DIBUJAR)['variable'].tolist()

fig, axes = plt.subplots(len(variables_a_dibujar), 2, figsize=(14, 3.5 * len(variables_a_dibujar)))
if len(variables_a_dibujar) == 1:
    axes = axes.reshape(1, -1)

for i, var in enumerate(variables_a_dibujar):
    plot_acf(df_sim[var], lags=n_lags_analisis, ax=axes[i, 0])
    axes[i, 0].set_title(f'ACF - {var}')
    plot_pacf(df_sim[var], lags=n_lags_analisis, ax=axes[i, 1], method='ywm')
    axes[i, 1].set_title(f'PACF - {var}')

plt.tight_layout()
plt.show()

## 4bis. ¿Sobre qué señales merece la pena calcular lags/rolling/diff?

No hace falta adivinarlo ni calcularlo aparte -- ya tienes todo lo necesario en las secciones anteriores. Combinamos dos criterios ya calculados:

- **PACF en lag 1** (sección 4): si es bajo, la señal no tiene estructura temporal de corto plazo que un lag pueda explotar -- meterle lags es ruido, no información.
- **Fuerza de tendencia / deriva** (sección 3, `cambio_total_vs_rango`): si es baja, la señal es plana en promedio y un rolling mean o diff no va a aportar nada que el valor puntual no diera ya.

La combinación de ambos te da, señal por señal, si vale la pena calcular lags, rolling/diff, ambas cosas, o ninguna -- en vez de aplicar el mismo `n_lags=5` + rolling + diff a las 41 XMEAS por igual, que es lo que has estado haciendo hasta ahora (razonable como primer paso, pero no es necesario mantenerlo así).

In [ ]:
UMBRAL_PACF = 0.2        # por encima de esto, el lag 1 aporta señal real (ajusta según lo que veas en tus gráficos PACF)
UMBRAL_TENDENCIA = 0.3   # por encima de esto, hay deriva/inercia -> rolling o diff tienen sentido

filas_recomendacion = []
for var in variables_pacf:
    pacf_lag1 = abs(resultados_pacf[var]['pacf'][1])
    fila_tendencia = tendencias.loc[var] if var in tendencias.index else None
    fuerza_tendencia = fila_tendencia['cambio_total_vs_rango'] if fila_tendencia is not None else np.nan

    usar_lags = pacf_lag1 > UMBRAL_PACF
    usar_rolling = fuerza_tendencia > UMBRAL_TENDENCIA if not np.isnan(fuerza_tendencia) else False

    if usar_lags and usar_rolling:
        recomendacion = 'lags + rolling/diff'
    elif usar_lags:
        recomendacion = 'solo lags'
    elif usar_rolling:
        recomendacion = 'solo rolling/diff'
    else:
        recomendacion = 'ninguna -- variable ~ruido, sin estructura temporal explotable'

    filas_recomendacion.append({
        'señal': var, 'pacf_lag1': pacf_lag1, 'fuerza_tendencia': fuerza_tendencia,
        'usar_lags': usar_lags, 'usar_rolling_diff': usar_rolling, 'recomendación': recomendacion
    })

df_recomendacion_features = pd.DataFrame(filas_recomendacion).sort_values('pacf_lag1', ascending=False)

print(f"Resumen: {df_recomendacion_features['usar_lags'].sum()} de {len(df_recomendacion_features)} señales con lags recomendados")
print(f"         {df_recomendacion_features['usar_rolling_diff'].sum()} de {len(df_recomendacion_features)} señales con rolling/diff recomendados")
print(f"         {(df_recomendacion_features['recomendación'].str.startswith('ninguna')).sum()} señales sin estructura temporal explotable (candidatas a excluir del feature engineering)")
df_recomendacion_features

## 5. Feature engineering (reutilizamos la función ya construida)

Misma función que en el notebook anterior, con lags + rolling + diff, causales.

In [ ]:
def _lags_potencia(n_lags_max, base=2):
    lags = []
    l = 1
    while l < n_lags_max:
        lags.append(l)
        l *= base
    lags.append(n_lags_max)
    return sorted(set(lags))


def construir_features(df, variables_entrada, usar_lags=True, n_lags=3,
                        espaciado_lags='consecutivo', base_potencia=2,
                        usar_rolling_mean=False, ventana_mean=5,
                        usar_rolling_std=False, ventana_std=5,
                        usar_diff=False):
    df_out = df.copy()
    columnas_features = list(variables_entrada)

    if espaciado_lags == 'potencia':
        lista_lags = _lags_potencia(n_lags, base=base_potencia)
    else:
        lista_lags = list(range(1, n_lags + 1))

    # Acumulamos las columnas nuevas en un diccionario y las unimos con UN solo concat al
    # final, en vez de insertarlas una a una con df_out[col] = ... . Insertar columna a
    # columna fragmenta el DataFrame internamente (pandas avisa con PerformanceWarning) y
    # es notablemente más lento cuando esta función se llama muchas veces seguidas -- como
    # ocurre en construir_features_multi_sim, que la invoca una vez POR CADA simulación.
    nuevas_columnas = {}
    for var in variables_entrada:
        if usar_lags:
            for lag in lista_lags:
                col = f'{var}_lag_{lag}'
                nuevas_columnas[col] = df_out[var].shift(lag)
                columnas_features.append(col)
        if usar_rolling_mean:
            col = f'{var}_roll_mean_{ventana_mean}'
            nuevas_columnas[col] = df_out[var].shift(1).rolling(window=ventana_mean).mean()
            columnas_features.append(col)
        if usar_rolling_std:
            col = f'{var}_roll_std_{ventana_std}'
            nuevas_columnas[col] = df_out[var].shift(1).rolling(window=ventana_std).std()
            columnas_features.append(col)
        if usar_diff:
            col = f'{var}_diff'
            nuevas_columnas[col] = df_out[var].diff()
            columnas_features.append(col)

    df_out = pd.concat([df_out, pd.DataFrame(nuevas_columnas, index=df_out.index)], axis=1)

    return df_out, columnas_features


def evaluar_modelo(X, y, train_frac=0.7, n_estimators=150, max_depth=3, learning_rate=0.1):
    corte = int(len(X) * train_frac)
    X_train, X_test = X.iloc[:corte], X.iloc[corte:]
    y_train, y_test = y.iloc[:corte], y.iloc[corte:]

    modelo = xgb.XGBRegressor(
        n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate,
        random_state=42, n_jobs=-1
    )
    modelo.fit(X_train, y_train)
    preds = modelo.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    return {'modelo': modelo, 'rmse': rmse, 'X_test': X_test, 'y_test': y_test,
            'preds': preds, 'corte': corte}

## 6. Modelo predictivo de la variable de calidad

Antes de recomendar nada, necesitas un modelo que prediga bien la variable de calidad a partir de las variables manipuladas y medidas. **Este modelo es el corazón de todo lo que viene después** — si predice mal, cualquier recomendación que construyas encima es ruido con apariencia de decisión informada.

Ojo: aquí metemos también las XMEAS intermedias como *inputs*, no solo las XMV. Esto es una decisión de diseño explícita — asumimos que en el momento de "recomendar" ya conoces el estado actual del proceso (temperaturas, presiones), y lo que decides es el próximo ajuste de XMV. Si tu caso real es distinto (recomendar sin conocer el estado intermedio), tendrías que quitar las XMEAS de la lista de entrada.

In [ ]:
variables_entrada_calidad = VARIABLES_XMV + VARIABLES_XMEAS

df_feat, cols_feat = construir_features(
    df_sim, variables_entrada_calidad,
    usar_lags=True, n_lags=5, espaciado_lags='consecutivo',
    usar_rolling_mean=True, ventana_mean=5,
    usar_rolling_std=False,
    usar_diff=True
)
df_feat[VARIABLE_CALIDAD] = df_sim[VARIABLE_CALIDAD].values
df_feat = df_feat.dropna().reset_index(drop=True)

X_calidad = df_feat[cols_feat]
y_calidad = df_feat[VARIABLE_CALIDAD]

res_calidad = evaluar_modelo(X_calidad, y_calidad, n_estimators=200, max_depth=4)
print(f"RMSE prediciendo {VARIABLE_CALIDAD}: {res_calidad['rmse']:.4f}")
print(f"Rango real de la variable: [{y_calidad.min():.2f}, {y_calidad.max():.2f}]  (desviación típica: {y_calidad.std():.2f})")

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(res_calidad['y_test'].values, label='Real', color='blue', alpha=0.6)
plt.plot(res_calidad['preds'], label='Predicción', color='red', linestyle='dashed', alpha=0.8)
plt.title(f"Modelo de calidad — RMSE {res_calidad['rmse']:.3f}")
plt.xlabel('Muestras de test')
plt.ylabel(VARIABLE_CALIDAD)
plt.legend()
plt.tight_layout()
plt.show()

xgb.plot_importance(res_calidad['modelo'], height=0.5, importance_type='weight', max_num_features=12)
plt.title('Importancia de variables — modelo de calidad')
plt.tight_layout()
plt.show()

## 6bis. Entrenar con más simulaciones (400 train / 100 test) -- hecho bien, no solo "más filas"

Con 500 simulaciones disponibles, usar más de una para entrenar es razonable -- un modelo entrenado sobre una sola simulación (`simulationRun == 1`) solo ha visto una "historia" del proceso, y no sabes si generaliza a las otras 499.

**Pero hay una decisión de diseño que hay que tomar con cuidado, no dar por hecha:** ¿qué significa "train" y "test" cuando la unidad natural no es una fila, es una simulación completa?

- **Split por simulación completa (lo correcto aquí):** 400 simulaciones enteras para entrenar, 100 simulaciones enteras -- nunca vistas, ni una fila -- para test. El modelo se evalúa en "historias" del proceso que no vio en absoluto.
- **Concatenar las 500 simulaciones en una serie larga y partir cronológicamente (lo que haría el código actual sin cambios):** esto es un error, no una simplificación aceptable. `construir_features` usa `shift()`/`rolling()`, que al concatenar simulaciones distintas calcularía lags falsos en cada una de las 499 costuras -- la primera fila de la simulación 37 tomaría como "lag 1" la última fila de la simulación 36, que es un experimento completamente distinto sin continuidad temporal real.

La solución: calcular las features **por separado dentro de cada simulación** (para que ningún lag cruce la frontera entre experimentos) y solo después concatenar y hacer el split por simulación completa.

Sobre el tiempo: entrenar con ~200.000 filas y ~50 features no es lento (segundos, no minutos) -- XGBoost escala bien en el número de filas. Lo que sí hay que hacer con cuidado es el cálculo de features (una llamada a `construir_features` por cada una de las 500 simulaciones), que este notebook resuelve con un bucle simple, no con algo costoso.

In [ ]:
def construir_features_multi_sim(df_raw, columna_sim, variables_entrada, columna_objetivo=None,
                                   **kwargs_features):
    """
    Aplica construir_features() de forma INDEPENDIENTE a cada simulación (evita que los
    lags/rolling/diff crucen la frontera entre simulaciones distintas) y concatena el resultado.

    Si se pasa columna_objetivo, se añade DENTRO del bucle por cada grupo -- así queda
    perfectamente alineada con el resto de columnas sin depender de reconstruir índices
    después de concatenar (que es donde es fácil desalinearse sin que salte ningún error).
    """
    partes = []
    columnas_features = None
    for sim_id, grupo in df_raw.groupby(columna_sim):
        grupo = grupo.sort_values(COLUMNA_TIEMPO).reset_index(drop=True)
        df_feat_sim, cols = construir_features(grupo, variables_entrada, **kwargs_features)
        if columna_objetivo is not None:
            df_feat_sim[columna_objetivo] = grupo[columna_objetivo].values
        df_feat_sim[columna_sim] = sim_id
        partes.append(df_feat_sim)
        columnas_features = cols
    df_concat = pd.concat(partes, ignore_index=True)
    return df_concat, columnas_features


def evaluar_modelo_split_por_simulacion(df_feat, cols_feat, columna_objetivo, columna_sim,
                                          simulaciones_train, simulaciones_test,
                                          n_estimators=150, max_depth=3, learning_rate=0.1):
    """
    Igual que evaluar_modelo(), pero el split es por SIMULACIÓN COMPLETA, no cronológico
    sobre una serie concatenada -- el test usa simulaciones que el modelo no vio ni una fila.
    """
    df_train = df_feat[df_feat[columna_sim].isin(simulaciones_train)]
    df_test = df_feat[df_feat[columna_sim].isin(simulaciones_test)]

    X_train, y_train = df_train[cols_feat], df_train[columna_objetivo]
    X_test, y_test = df_test[cols_feat], df_test[columna_objetivo]

    modelo = xgb.XGBRegressor(
        n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate,
        random_state=42, n_jobs=-1
    )
    modelo.fit(X_train, y_train)
    preds = modelo.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    return {"modelo": modelo, "rmse": rmse, "X_test": X_test, "y_test": y_test, "preds": preds}

In [ ]:
import time

N_SIM_TRAIN = 400
N_SIM_TEST = 100

todas_las_simulaciones = sorted(df_raw['simulationRun'].unique())
assert len(todas_las_simulaciones) >= N_SIM_TRAIN + N_SIM_TEST, f"Solo hay {len(todas_las_simulaciones)} simulaciones disponibles"

simulaciones_train = todas_las_simulaciones[:N_SIM_TRAIN]
simulaciones_test = todas_las_simulaciones[N_SIM_TRAIN:N_SIM_TRAIN + N_SIM_TEST]

t0 = time.time()
df_feat_multi, cols_feat_multi = construir_features_multi_sim(
    df_raw, 'simulationRun', variables_entrada_calidad, columna_objetivo=VARIABLE_CALIDAD,
    usar_lags=True, n_lags=5, espaciado_lags='consecutivo',
    usar_rolling_mean=True, ventana_mean=5,
    usar_rolling_std=False,
    usar_diff=True
)
df_feat_multi = df_feat_multi.dropna().reset_index(drop=True)
t1 = time.time()
tiempo_features = t1 - t0
print(f"Features calculadas por simulación (500 simulaciones), tiempo: {tiempo_features:.2f}s")
print(f"Filas totales tras construir features y dropna: {len(df_feat_multi)}")

t0 = time.time()
res_multi_sim = evaluar_modelo_split_por_simulacion(
    df_feat_multi, cols_feat_multi, VARIABLE_CALIDAD, 'simulationRun',
    simulaciones_train, simulaciones_test,
    n_estimators=200, max_depth=4
)
t1 = time.time()
tiempo_entrenamiento = t1 - t0

rmse_multi = res_multi_sim['rmse']
rmse_una_sim = res_calidad['rmse']

print(f"\nEntrenamiento ({len(simulaciones_train)} simulaciones) + evaluación ({len(simulaciones_test)} simulaciones): {tiempo_entrenamiento:.2f}s")
print(f"RMSE con split por simulación completa: {rmse_multi:.4f}")
print(f"Para comparar -- RMSE con solo simulationRun==1: {rmse_una_sim:.4f}")

## 7. Forecasting de la calidad — qué va a pasar si no tocas nada

Hasta ahora, `recomendar_ajuste` parte siempre del **último estado observado** del proceso. Eso tiene una limitación: si el proceso ya está en movimiento hacia un estado peor (por ejemplo, la temperatura llevaba varios pasos subiendo), recomendar sobre el estado *actual* ignora hacia dónde se dirige el proceso *ya*, con o sin tu intervención.

La idea aquí es simple y deliberadamente separada del control (todavía no tocamos XMV): entrenamos un modelo que predice `VARIABLE_CALIDAD` a un horizonte `h` pasos vista, **dejando las XMV fijas en su último valor observado** — es decir, "¿qué va a pasar si el operador no cambia nada?". Ese es el mismo patrón de forecasting que usamos en el notebook de la bomba, aplicado aquí a la variable de calidad del TEP.

**Importante — qué es y qué no es esto:** esto NO es todavía control predictivo (MPC). No estamos simulando "qué pasaría si cambio la XMV X" — solo estamos prediciendo la inercia natural del proceso. El resultado de esta sección se va a usar más adelante como **punto de partida alternativo** para la búsqueda de recomendación, en vez del último dato observado. El salto a "qué pasaría si cambio la XMV" (control predictivo real) es un notebook aparte, porque requiere encadenar predicciones de forma recursiva y ahí el error se acumula paso a paso — no es una extensión trivial de esto.

In [ ]:
HORIZONTE_FORECAST = 5  # nº de muestras hacia adelante (con 'sample' cada 3 min, 5 = 15 minutos vista)

# Mismas features de entrada que el modelo de nowcast -- la única diferencia es el target,
# que aquí desplazamos h pasos hacia el futuro en vez de usar el valor en el mismo instante.
df_feat_fc, cols_feat_fc = construir_features(
    df_sim, variables_entrada_calidad,
    usar_lags=True, n_lags=5, espaciado_lags='consecutivo',
    usar_rolling_mean=True, ventana_mean=5,
    usar_rolling_std=False,
    usar_diff=True
)
df_feat_fc[f'target_futuro_{VARIABLE_CALIDAD}'] = df_sim[VARIABLE_CALIDAD].shift(-HORIZONTE_FORECAST).values
# Guardamos también el valor ACTUAL (sin desplazar) en la misma construcción, para que el
# dropna() de abajo lo arrastre con exactamente los mismos índices que el resto de columnas
# -- así el baseline naive de la siguiente celda queda perfectamente alineado por diseño,
# sin tener que reconstruir el recorte de índices a mano (que es donde es fácil desalinearse).
df_feat_fc['valor_actual_calidad'] = df_sim[VARIABLE_CALIDAD].values
df_feat_fc = df_feat_fc.dropna().reset_index(drop=True)

X_forecast = df_feat_fc[cols_feat_fc]
y_forecast = df_feat_fc[f'target_futuro_{VARIABLE_CALIDAD}']

res_forecast = evaluar_modelo(X_forecast, y_forecast, n_estimators=200, max_depth=4)
print(f"RMSE forecasting {VARIABLE_CALIDAD} a +{HORIZONTE_FORECAST} pasos: {res_forecast['rmse']:.4f}")
print(f"Para comparar -- RMSE de nowcast (horizonte 0): {res_calidad['rmse']:.4f}")
print(f"(Es normal y esperable que el RMSE de forecasting sea mayor: predices algo que aún no ha pasado)")

### ¿RMSE=3.3 con std=5.97 es bueno? -- la comparación correcta no es esa

Comparar el RMSE directamente contra la desviación típica de la señal responde a una pregunta distinta de la que importa. La std mide cuánto varía la señal en **todo** su rango histórico; tu modelo no necesita explicar esa variabilidad completa, necesita predecir mejor que la alternativa más simple posible.

El benchmark correcto para forecasting es un **baseline naive (persistence model)**: predecir que el valor en `t+h` va a ser igual al último valor conocido en `t`. Si tu XGBoost con todo el feature engineering apenas le gana a "no va a cambiar nada", el modelo no está aportando información real más allá de la inercia del proceso que ya conocías por el PACF. Si le gana con margen claro, ahí sí tienes evidencia de que está aprendiendo algo genuino.

In [ ]:
# Baseline naive: predice que el valor en t+h será igual al valor en t (persistence model)
# Se calcula sobre el MISMO conjunto de test que el modelo, para que la comparación sea justa.
# Usamos "valor_actual_calidad", que se guardó DENTRO de df_feat_fc antes del dropna() --
# por construcción tiene exactamente los mismos índices que y_forecast, sin necesidad de
# reconstruir el recorte a mano (que es donde es fácil desalinearse sin que salte ningún error).
corte_fc = res_forecast["corte"]
valor_actual_test = df_feat_fc["valor_actual_calidad"].iloc[corte_fc:].reset_index(drop=True)

y_test_fc = res_forecast["y_test"].reset_index(drop=True)
rmse_naive = np.sqrt(mean_squared_error(y_test_fc, valor_actual_test))
rmse_modelo = res_forecast["rmse"]

print(f"RMSE del modelo XGBoost (forecasting +{HORIZONTE_FORECAST}):  {rmse_modelo:.4f}")
print(f"RMSE del baseline naive (no cambia nada):     {rmse_naive:.4f}")

mejora_pct = (1 - rmse_modelo / rmse_naive) * 100
print(f"\nMejora del modelo sobre el baseline naive: {mejora_pct:+.1f}%")

if mejora_pct < 5:
    print("ALERTA: El modelo apenas mejora sobre no va a cambiar nada -- gran parte de tu RMSE bajo")
    print("    puede deberse a que el proceso es lento/estable en este horizonte, no a que el modelo")
    print("    esté aprendiendo relaciones causales útiles. Revisa si el horizonte es demasiado corto")
    print("    para que el forecasting aporte algo que el propio valor actual no diera ya.")
elif mejora_pct < 20:
    print("El modelo mejora sobre el baseline, pero de forma modesta -- hay margen para revisar")
    print("features, horizonte, o aceptar que el proceso tiene poca variabilidad predecible aquí.")
else:
    print("El modelo mejora claramente sobre el baseline naive -- señal de que está capturando")
    print("  relaciones reales entre las XMV/XMEAS y la calidad futura, no solo inercia.")

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(res_forecast['y_test'].values, label='Real', color='blue', alpha=0.6)
plt.plot(res_forecast['preds'], label='Predicción', color='red', linestyle='dashed', alpha=0.8)
plt.title(f"Forecasting a +{HORIZONTE_FORECAST} pasos — RMSE {res_forecast['rmse']:.3f}")
plt.xlabel('Muestras de test')
plt.ylabel(f'{VARIABLE_CALIDAD} (futuro)')
plt.legend()
plt.tight_layout()
plt.show()

## 7bis. Forecasting entrenado con todo el dataset (400 train / 100 test, por simulación)

Mismo criterio que en nowcast (sección 6bis): entrenar con una sola simulación da una estimación optimista y poco fiable, porque el modelo puede estar memorizando las particularidades de esa "historia" concreta en vez de aprender la dinámica general del proceso.

Aquí hay una fuga adicional a la que no basta con aplicar `construir_features_multi_sim` tal cual: el **target** de forecasting (`shift(-h)`, el valor h pasos en el futuro) también tiene que calcularse **dentro de cada simulación por separado**. Si se calcula sobre el dataframe ya concatenado, las últimas `h` filas de cada simulación tomarían como "futuro" datos de la simulación siguiente -- que es un experimento completamente distinto, no una continuación real. Es la misma fuga que ya evitamos para los lags, aplicada ahora al target en vez de a los inputs.

In [ ]:
def construir_features_forecast_multi_sim(df_raw, columna_sim, variables_entrada, columna_objetivo,
                                             horizonte, **kwargs_features):
    """
    Igual que construir_features_multi_sim, pero además calcula el target desplazado
    (shift(-horizonte)) DENTRO de cada simulación -- evita que el target de las últimas
    filas de una simulación se cuele desde la simulación siguiente.
    """
    partes = []
    columnas_features = None
    nombre_target = f'target_futuro_{columna_objetivo}'
    for sim_id, grupo in df_raw.groupby(columna_sim):
        grupo = grupo.sort_values(COLUMNA_TIEMPO).reset_index(drop=True)
        df_feat_sim, cols = construir_features(grupo, variables_entrada, **kwargs_features)
        df_feat_sim[nombre_target] = grupo[columna_objetivo].shift(-horizonte).values
        df_feat_sim['valor_actual_calidad'] = grupo[columna_objetivo].values
        df_feat_sim[columna_sim] = sim_id
        partes.append(df_feat_sim)
        columnas_features = cols
    df_concat = pd.concat(partes, ignore_index=True)
    return df_concat, columnas_features, nombre_target

In [ ]:
t0 = time.time()
df_feat_fc_multi, cols_feat_fc_multi, nombre_target_fc = construir_features_forecast_multi_sim(
    df_raw, 'simulationRun', variables_entrada_calidad, VARIABLE_CALIDAD, HORIZONTE_FORECAST,
    usar_lags=True, n_lags=5, espaciado_lags='consecutivo',
    usar_rolling_mean=True, ventana_mean=5,
    usar_rolling_std=False,
    usar_diff=True
)
df_feat_fc_multi = df_feat_fc_multi.dropna().reset_index(drop=True)
t1 = time.time()
print(f"Features de forecasting calculadas por simulación: {t1-t0:.2f}s")
print(f"Filas totales tras construir features y dropna: {len(df_feat_fc_multi)}")

t0 = time.time()
res_forecast_multi = evaluar_modelo_split_por_simulacion(
    df_feat_fc_multi, cols_feat_fc_multi, nombre_target_fc, 'simulationRun',
    simulaciones_train, simulaciones_test,
    n_estimators=200, max_depth=4
)
t1 = time.time()

rmse_forecast_multi = res_forecast_multi['rmse']
rmse_forecast_una_sim = res_forecast['rmse']

print(f"\nEntrenamiento + evaluación: {t1-t0:.2f}s")
print(f"RMSE forecasting +{HORIZONTE_FORECAST} con split por simulación completa: {rmse_forecast_multi:.4f}")
print(f"Para comparar -- RMSE forecasting +{HORIZONTE_FORECAST} con solo simulationRun==1: {rmse_forecast_una_sim:.4f}")

### Real vs predicho, sobre las 100 simulaciones de test

Para verlo "en predicción" como pediste: aquí no hay una sola serie continua de test (son 100 simulaciones distintas concatenadas), así que el gráfico de líneas real-vs-predicho tiene sentido por tramos, no como una sola curva continua -- cada simulación de test es independiente de la siguiente. Mostramos primero un scatter (real vs predicho, la forma estándar de evaluar esto cuando el test no es una serie temporal única) y después una muestra de una sola simulación de test a modo de ejemplo visual de la forma.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter real vs predicho -- la forma correcta de ver el ajuste cuando el test
# son 100 simulaciones independientes, no una sola serie continua
axes[0].scatter(res_forecast_multi['y_test'], res_forecast_multi['preds'], alpha=0.15, s=8, color='steelblue')
lims = [res_forecast_multi['y_test'].min(), res_forecast_multi['y_test'].max()]
axes[0].plot(lims, lims, color='crimson', linestyle='--', label='predicción perfecta')
axes[0].set_xlabel('Real')
axes[0].set_ylabel('Predicho')
axes[0].set_title(f'Real vs predicho -- {len(simulaciones_test)} simulaciones de test\nRMSE={rmse_forecast_multi:.4f}')
axes[0].legend()

# Ejemplo visual: una sola simulación de test, para ver la forma temporal de la predicción
sim_ejemplo = simulaciones_test[0]
mask_ejemplo = df_feat_fc_multi.loc[res_forecast_multi['X_test'].index, 'simulationRun'] == sim_ejemplo
axes[1].plot(res_forecast_multi['y_test'][mask_ejemplo].values, label='Real', color='blue', alpha=0.7)
axes[1].plot(res_forecast_multi['preds'][mask_ejemplo], label='Predicción', color='red', linestyle='dashed', alpha=0.8)
axes[1].set_title(f'Ejemplo: simulación {sim_ejemplo} (una de las 100 de test)')
axes[1].set_xlabel('Muestras dentro de esa simulación')
axes[1].legend()

plt.tight_layout()
plt.show()

### Usar el estado proyectado como punto de partida de la recomendación

Con el modelo de forecasting ya entrenado, podemos predecir *ahora* cuál será previsiblemente la calidad dentro de `HORIZONTE_FORECAST` pasos si nadie toca nada. Eso por sí solo ya es información útil para un operador ("si no haces nada, la calidad va a caer"), y además nos da pie a comparar dos formas de recomendar:

- **Recomendación reactiva** (la que ya tenías): busca el mejor ajuste de XMV partiendo del **último estado observado**.
- **Recomendación anticipativa**: busca el mejor ajuste de XMV partiendo del **estado que el proceso tendrá dentro de `h` pasos si no intervienes** — es decir, se adelanta a la deriva natural del proceso en vez de reaccionar solo a lo que ya pasó.

Ojo: seguimos sin simular qué pasaría *con* la intervención a lo largo de esos `h` pasos — solo cambiamos el punto de partida de la búsqueda. Es una mejora real, pero modesta comparada con MPC.

In [ ]:
calidad_actual = df_sim[VARIABLE_CALIDAD].iloc[-1]
calidad_proyectada = res_forecast['modelo'].predict(df_feat_fc[cols_feat_fc].iloc[[-1]])[0]

print(f"Calidad actual (último dato observado):              {calidad_actual:.4f}")
print(f"Calidad proyectada a +{HORIZONTE_FORECAST} pasos (sin intervención): {calidad_proyectada:.4f}")

diferencia = calidad_proyectada - calidad_actual
if abs(diferencia) > res_forecast['rmse']:
    tendencia = "empeorando" if diferencia < 0 else "mejorando"
    print(f"\n⚠️  El proceso parece estar {tendencia} por su cuenta (diferencia mayor que el error típico del modelo).")
    print("   Puede tener sentido recomendar sobre el estado proyectado, no solo sobre el actual.")
else:
    print("\nLa diferencia entre el estado actual y el proyectado es pequeña frente al error del modelo -- el proceso está razonablemente estable, no hay una deriva clara que anticipar.")

## 8. De predicción a recomendación

Aquí está la pieza nueva. La idea:

1. Tomamos el **estado actual** del proceso (última fila de datos: lags, rolling, etc. ya calculados salvo las XMV, que son lo que queremos decidir).
2. Probamos una **rejilla de combinaciones posibles de XMV**, dentro del rango que el proceso ya ha mostrado en los datos históricos (nunca fuera — el árbol no sabe extrapolar).
3. Para cada combinación, reconstruimos el vector de features y le pedimos al modelo que prediga la calidad resultante.
4. Nos quedamos con la combinación que maximiza (o minimiza, según el objetivo) la calidad predicha.

### La advertencia que de verdad importa aquí

Esto **no es optimización real de proceso** — es "probar muchas combinaciones ya vistas y quedarte con la mejor según lo que el modelo cree". Si el óptimo real está fuera del rango histórico, este método nunca lo va a encontrar, y si lo fuerzas a buscar fuera de rango, el modelo va a inventarse una predicción sin ninguna base real. Por eso la rejilla está anclada estrictamente al min/max observado — es una salvaguarda, no un capricho.

In [ ]:
def construir_vector_estado_actual(df_raw, df_feat, cols_feat, variables_entrada, idx=-1):
    """
    Toma la última fila disponible de df_feat (con todos los lags/rolling/diff ya calculados)
    como snapshot del estado del proceso, para usarla como plantilla en la búsqueda.
    """
    return df_feat[cols_feat].iloc[[idx]].copy()


def recomendar_ajuste(modelo, df_raw, df_feat, cols_feat, variables_xmv, variable_calidad,
                       modo='rejilla', n_puntos_por_variable=5, n_muestras_aleatorias=2000,
                       objetivo='maximizar', seed=42, estado_base=None):
    """
    Busca la mejor combinación de XMV, ACOTADA al rango histórico observado (nunca extrapola).

    modo='rejilla': malla completa (itertools.product). Exhaustivo pero crece como
                     n_puntos_por_variable ** n_variables -- inviable con muchas variables
                     (con 11 XMV y 5 puntos/variable son ~49 millones de combinaciones).
                     Útil solo con pocas variables (hasta 4-5) o pocos puntos por variable.

    modo='aleatorio': muestrea n_muestras_aleatorias combinaciones al azar dentro del rango
                       válido de cada variable. El coste NO depende del número de variables,
                       solo del número de muestras que decidas pagar -- por eso escala a
                       cualquier número de XMV sin explotar en memoria ni en tiempo.
                       En espacios de muchas dimensiones, random search suele acercarse al
                       óptimo de grid search con una fracción ínfima del coste, porque la
                       mayoría de combinaciones de una malla completa son redundantes.

    estado_base: si se pasa (un DataFrame de 1 fila con las columnas cols_feat), se usa como
                 punto de partida en vez del último dato observado -- por ejemplo, el estado
                 PROYECTADO a h pasos vista con el modelo de forecasting (sección 6), para una
                 recomendación anticipativa en vez de puramente reactiva.

    IMPORTANTE: en ambos modos, todas las combinaciones se construyen de golpe en un único
    DataFrame y se evalúan con UNA sola llamada a modelo.predict() (vectorizado), no una
    llamada por combinación -- eso es lo que evita el overhead fijo de predict() multiplicado
    por miles/millones de veces, que es la causa más común de lentitud aquí (no el tamaño
    del modelo, ni la falta de GPU).
    """
    if estado_base is None:
        estado_base = construir_vector_estado_actual(df_raw, df_feat, cols_feat, variables_xmv)
    rng = np.random.default_rng(seed)

    rangos = {var: (df_raw[var].min(), df_raw[var].max()) for var in variables_xmv}

    if modo == 'rejilla':
        rejillas = {var: np.linspace(vmin, vmax, n_puntos_por_variable)
                    for var, (vmin, vmax) in rangos.items()}
        combinaciones = np.array(list(itertools.product(*[rejillas[v] for v in variables_xmv])))
    elif modo == 'aleatorio':
        combinaciones = np.column_stack([
            rng.uniform(rangos[var][0], rangos[var][1], n_muestras_aleatorias)
            for var in variables_xmv
        ])
    else:
        raise ValueError("modo debe ser 'rejilla' o 'aleatorio'")

    n_combos = len(combinaciones)

    # Construir TODAS las filas de una vez (vectorizado), en vez de una por iteración.
    # Las columnas de variables_xmv se preparan en un diccionario y se asignan con un
    # único .assign() (o reasignación conjunta), en vez de una asignación por columna en
    # un bucle -- evita fragmentar el DataFrame igual que en construir_features.
    filas = pd.concat([estado_base] * n_combos, ignore_index=True)
    columnas_xmv_nuevas = {
        var: combinaciones[:, j] for j, var in enumerate(variables_xmv) if var in filas.columns
    }
    filas = filas.assign(**columnas_xmv_nuevas)

    # UNA sola llamada a predict() para todas las combinaciones
    preds = modelo.predict(filas[cols_feat])

    df_resultados = pd.DataFrame(combinaciones, columns=variables_xmv)
    df_resultados['calidad_predicha'] = preds

    ascending = (objetivo == 'minimizar')
    df_resultados = df_resultados.sort_values('calidad_predicha', ascending=ascending).reset_index(drop=True)
    return df_resultados

### Ejecutar la recomendación

Buscamos la combinación de XMV que **maximiza** la calidad predicha (cambia `objetivo='minimizar'` si tu variable de calidad es del tipo "cuanto menos, mejor" — como sería el caso de acidez).

In [ ]:
# Con 11 variables XMV reales, modo='rejilla' con 5 puntos/variable son ~48.8 millones
# de combinaciones -- eso es lo que te mató el kernel. Con >= 5-6 variables, usa
# modo='aleatorio': el coste no depende de cuántas XMV tengas, solo de cuántas
# muestras decidas pagar (ver benchmark en la siguiente celda para comparar ambos
# modos con un tamaño de rejilla que SÍ es seguro, y ver por qué el real no lo era).
df_recomendaciones = recomendar_ajuste(
    modelo=res_calidad['modelo'],
    df_raw=df_raw,
    df_feat=df_feat,
    cols_feat=cols_feat,
    variables_xmv=VARIABLES_XMV,
    variable_calidad=VARIABLE_CALIDAD,
    modo='aleatorio',
    n_muestras_aleatorias=2000,
    objetivo='maximizar'
)

print("1 combinaciones recomendadas:")
df_recomendaciones.head(1)

### Por qué esto ya no "tarda una barbaridad": rejilla vs aleatorio, medido

Dos cosas cambiaron respecto a la primera versión: (1) `predict()` se llama **una sola vez** con todas las combinaciones juntas en vez de una vez por combinación, y (2) puedes elegir `modo='aleatorio'` para que el coste no explote con el número de variables. Vamos a medirlo con tus propios datos y modelo, no de oídas.

In [ ]:
import time

n_xmv = len(VARIABLES_XMV)
# Con muchas variables, n_puntos_por_variable pequeño para que la rejilla no reviente
# (5 puntos ^ 11 variables = 48.8M combinaciones -- eso es justo lo que mató el kernel arriba).
n_puntos_seguro = 2 if n_xmv >= 6 else 5

for modo, kwargs in [
    ('rejilla', {'n_puntos_por_variable': n_puntos_seguro}),
    ('aleatorio', {'n_muestras_aleatorias': 500}),
]:
    t0 = time.time()
    df_bench = recomendar_ajuste(
        modelo=res_calidad['modelo'], df_raw=df_raw, df_feat=df_feat, cols_feat=cols_feat,
        variables_xmv=VARIABLES_XMV, variable_calidad=VARIABLE_CALIDAD, modo=modo, objetivo='maximizar',
        **kwargs
    )
    t1 = time.time()
    mejor_calidad = df_bench.iloc[0]["calidad_predicha"]
    print(f"modo={modo:10s}  combinaciones={len(df_bench):8d}  tiempo={t1-t0:.4f}s  mejor_calidad={mejor_calidad:.4f}")

print()
print(f"Con tus {n_xmv} variables XMV reales:")
print(f"  modo=rejilla con 5 puntos/variable    -> {5**n_xmv:,} combinaciones  <- esto te mató el kernel arriba")
print(f"  modo=aleatorio con 2000 muestras       -> 2,000 combinaciones (mismo coste siempre, no depende de n_xmv)")

### Comparación: recomendación reactiva vs anticipativa

Usamos el estado proyectado (sección 6) como punto de partida y comparamos la recomendación resultante contra la que ya tenías (partiendo del último dato observado). Si el proceso está razonablemente estable, ambas deberían parecerse bastante -- una diferencia grande es la señal de que anticiparte al futuro sí está cambiando la decisión, no solo maquillándola.

In [ ]:
estado_proyectado = df_feat_fc[cols_feat_fc].iloc[[-1]].copy()

df_recomendaciones_anticipativa = recomendar_ajuste(
    modelo=res_calidad['modelo'],
    df_raw=df_raw,
    df_feat=df_feat,
    cols_feat=cols_feat,
    variables_xmv=VARIABLES_XMV,
    variable_calidad=VARIABLE_CALIDAD,
    modo='aleatorio',
    n_muestras_aleatorias=2000,
    objetivo='maximizar',
    estado_base=estado_proyectado
)

print("Recomendación reactiva (estado actual):")
print(df_recomendaciones.iloc[0][VARIABLES_XMV + ['calidad_predicha']])
print()
print(f"Recomendación anticipativa (estado proyectado a +{HORIZONTE_FORECAST} pasos):")
print(df_recomendaciones_anticipativa.iloc[0][VARIABLES_XMV + ['calidad_predicha']])

## 9. Comprobación de fiabilidad — ¿la recomendación está dentro de lo que el modelo realmente conoce?

Esta celda es la que convierte "una recomendación" en "una recomendación en la que puedes confiar un poco más". Comprobamos si la mejor combinación encontrada corresponde a una zona del espacio de XMV donde hubo **suficientes datos históricos parecidos** — si la mejor recomendación vive en una esquina del espacio con pocos puntos de entrenamiento cerca, es una señal de alerta, no una señal de éxito.

In [ ]:
from scipy.spatial import cKDTree

def distancia_a_datos_historicos(mejor_combo, df_raw, variables_xmv):
    """
    Distancia (normalizada) de la combinación recomendada al punto histórico más cercano,
    usando un KDTree en vez de una matriz de distancias completa (cdist).

    Con un histórico grande (aquí, ~250.000 filas si df_raw son las 500 simulaciones
    completas), calcular todas las distancias contra todas las filas no es necesario --
    solo buscamos EL vecino más cercano, que es justo lo que un KDTree resuelve sin
    construir esa matriz gigante en memoria.
    """
    datos_hist = df_raw[variables_xmv].values
    rangos = datos_hist.max(axis=0) - datos_hist.min(axis=0)
    rangos[rangos == 0] = 1
    datos_hist_norm = datos_hist / rangos
    combo_norm = np.array(mejor_combo) / rangos

    tree = cKDTree(datos_hist_norm)
    dist_min, _ = tree.query(combo_norm)
    return dist_min

mejor_combo = df_recomendaciones.iloc[0][VARIABLES_XMV].values
dist_min = distancia_a_datos_historicos(mejor_combo, df_raw, VARIABLES_XMV)

# Referencia: distancia típica entre puntos históricos consecutivos, para dar contexto a la cifra.
# Aquí NO usamos cdist (que compararía todas las filas contra todas -> misma explosión de
# memoria que arriba). Solo queremos la distancia fila[i] vs fila[i+1], una comparación
# uno-a-uno, así que una resta vectorizada basta y es muchísimo más barata.
datos_hist_norm_ref = df_raw[VARIABLES_XMV].values / (df_raw[VARIABLES_XMV].max() - df_raw[VARIABLES_XMV].min()).values
dist_referencia = np.median(np.linalg.norm(datos_hist_norm_ref[1:] - datos_hist_norm_ref[:-1], axis=1))

print(f"Mejor combinación recomendada: {dict(zip(VARIABLES_XMV, mejor_combo))}")
calidad_predicha_top = df_recomendaciones.iloc[0]["calidad_predicha"]
print(f"Calidad predicha: {calidad_predicha_top:.4f}")
print(f"Distancia (normalizada) al dato histórico más cercano: {dist_min:.4f}")
print(f"Distancia típica de referencia entre muestras consecutivas: {dist_referencia:.4f}")

if dist_min > 3 * dist_referencia:
    print("\n⚠️  ALERTA: esta recomendación está lejos de cualquier combinación vista en los datos.")
    print("   El modelo está extrapolando -- trátala como hipótesis a validar, no como instrucción a aplicar.")
else:
    print("\n✓ La recomendación está razonablemente cerca de zonas ya observadas en los datos históricos.")

## 10. Qué falta para que esto sea un sistema de recomendación real (no un ejercicio)

Deja esto anotado para cuando tengas los datos reales de la almazara, porque son las piezas que este notebook simplifica a propósito para que puedas centrarte en la arquitectura:

- **La variable de calidad real puede llegar con retardo y a otra cadencia** (análisis de laboratorio vs sensores en línea). Aquí la calidad se genera al mismo ritmo que el resto — en tu caso probablemente no será así, y tendrás que resolver un forecasting de horizonte largo antes de poder recomendar nada.
- **La búsqueda en rejilla no escala bien con muchas variables manipulables.** Con 3 XMV y 5 puntos por variable ya son 125 combinaciones; con 11 XMV (como en el TEP real) serían millones. Para eso existen métodos de optimización más eficientes (optimización bayesiana, algoritmos genéticos) — pero no los necesitas para aprender el concepto.
- **El modelo no sabe nada de restricciones de seguridad o de proceso** (p. ej. "la presión no puede superar X"). Cualquier recomendación real necesita una capa de reglas duras por encima del modelo, no solo el óptimo estadístico.
- **La comprobación de distancia a datos históricos es una heurística simple**, no una medida de incertidumbre real del modelo. Para algo más riguroso, XGBoost puede combinarse con estimación de incertidumbre (ensembles, quantile regression) — es un paso natural una vez domines esta versión básica.
- **Sobre GPU**: acelera el *entrenamiento* de XGBoost (`tree_method='hist', device='cuda'`) sobre datasets grandes, y `predict()` vectorizado sobre muchas filas a la vez. No ayuda si el cuello de botella es cómo está escrito el bucle de búsqueda (llamadas individuales a predict, o generar una malla que no cabe en memoria) — eso se arregla vectorizando y con `modo='aleatorio'`, como en este notebook, no añadiendo hardware. Si algún día entrenas con millones de filas y el *entrenamiento* (no la recomendación) es lo lento, ahí sí GPU tiene sentido.
- **Si más adelante necesitas exprimir más la búsqueda que random search**, el siguiente escalón es optimización bayesiana (p. ej. librería `scikit-optimize` o `optuna`): en vez de muestrear a ciegas, usa los resultados ya evaluados para decidir qué combinación probar después, así que converge a un buen óptimo con muchas menos llamadas al modelo que random search. No lo necesitas todavía con pocas variables, pero es la pieza que te faltaría si el espacio de búsqueda creciera mucho más o cada evaluación del modelo fuera cara.